# 01 Data cleaning
Input: fixed-schema `network_traffic.csv`. Output: clean flows and cleaning reports.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import config as cfg

for folder in (cfg.DATA_PROCESSED, cfg.MODELS, cfg.REPORTS): folder.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

## Load and validate
Read the known schema directly; retain row IDs for traceability.

In [ ]:
df_raw = pd.read_csv(cfg.RAW_PATH)
df = df_raw.copy()
df.insert(0, "row_id", np.arange(len(df)))
log = {"raw_file": cfg.RAW_FILE, "raw_rows": int(len(df_raw)), "raw_columns": int(df_raw.shape[1])}
assert sorted(df[cfg.LABEL].dropna().unique().tolist()) == [0, 1]
n_inf = int(np.isinf(df[cfg.NUMERIC_COLS].to_numpy(dtype=float)).sum())
df[cfg.NUMERIC_COLS] = df[cfg.NUMERIC_COLS].replace([np.inf, -np.inf], np.nan)
log["infinite_values_set_to_nan"] = n_inf
rows_no_label = int(df[cfg.LABEL].isna().sum())
df = df.dropna(subset=[cfg.LABEL]).copy()
log["rows_dropped_missing_label"] = rows_no_label
feature_cols = [c for c in df.columns if c != "row_id"]
dup_mask = df.duplicated(subset=feature_cols, keep="first")
n_dup = int(dup_mask.sum())
df = df.loc[~dup_mask].copy()
log["exact_duplicates_removed"] = n_dup

## Invalid values and outliers
Invalid cells become missing. Outliers are reported by class and never removed.

In [ ]:
invalid = {}
for c in [cfg.DURATION, cfg.PACKETS, cfg.BYTES_SENT, cfg.BYTES_RECEIVED, cfg.BYTES_PER_PACKET]:
    bad = pd.to_numeric(df[c], errors="coerce") < 0
    invalid[c] = int(bad.sum()); df.loc[bad, c] = np.nan
for c in [cfg.SRC_PORT, cfg.DST_PORT]:
    values = pd.to_numeric(df[c], errors="coerce")
    bad = (values < 0) | (values > 65535)
    invalid[c] = int(bad.sum()); df.loc[bad, c] = np.nan
for c in [cfg.SRC_IP, cfg.DST_IP]:
    values = pd.to_numeric(df[c], errors="coerce")
    bad = (values < 0) | (values > 4294967295)
    invalid[c] = int(bad.sum()); df.loc[bad, c] = np.nan
log["invalid_cells_set_to_nan"] = invalid

rows = []
for c in cfg.NUMERIC_COLS:
    s = pd.to_numeric(df[c], errors="coerce")
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    out = (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)
    rows.append({"column": c, "outliers": int(out.sum()), "share": float(out.mean()),
                 "share_in_normal": float(out[df[cfg.LABEL] == 0].mean()),
                 "share_in_anomalous": float(out[df[cfg.LABEL] == 1].mean())})
pd.DataFrame(rows).sort_values("share", ascending=False).to_csv(cfg.REPORTS / "outlier_summary.csv", index=False)
sns.countplot(x=cfg.LABEL, data=df); plt.title("Class distribution after cleaning"); plt.show()

## Save
Outputs: `flows_clean.csv`, `cleaning_log.json`, and `outlier_summary.csv`.

In [ ]:
df.to_csv(cfg.DATA_PROCESSED / "flows_clean.csv", index=False)
counts = df[cfg.LABEL].value_counts().sort_index()
log.update({"class_counts": {int(k): int(v) for k, v in counts.items()}, "clean_rows": int(len(df)),
            "clean_columns": int(df.shape[1]), "rows_removed_total": int(len(df_raw) - len(df))})
(cfg.REPORTS / "cleaning_log.json").write_text(json.dumps(log, indent=2), encoding="utf-8")
print(df.head(), df.shape)
print(json.dumps(log, indent=2))